# AURALIS: train the obstacle detector (YOLO)

**Runtime → Change runtime type → T4 GPU**, then run each cell from top to bottom. It takes about 1 hour.

Full guide: `docs/DATASET_GUIDE.md`

In [ ]:
!pip -q install ultralytics roboflow
!git clone -q https://github.com/Lakshya0104/Auralis-.git
%cd Auralis-
!nvidia-smi --query-gpu=name --format=csv,noheader

## 1. Download datasets from Roboflow

On each Roboflow Universe dataset page, click **Download Dataset → YOLOv8 → show download code**. Paste the snippets below, one block per dataset. Your **own labelled cane photos** go here too.

Each `dataset.location` is a folder, and they're collected in `sources`.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_KEY")
sources = []

# --- paste one block per dataset, e.g. ---
# project = rf.workspace("some-workspace").project("pothole-detection")
# dataset = project.version(1).download("yolov8")
# sources.append(dataset.location)

# --- your own photos (Roboflow project with AURALIS class names) ---
# project = rf.workspace("your-workspace").project("auralis-cane")
# dataset = project.version(1).download("yolov8")
# sources.append(dataset.location)

print(sources)

**Kaggle instead?** Upload `kaggle.json` and run `!kaggle datasets download -d <owner/name> --unzip -p data/<name>`, then `sources.append('data/<name>')`. The folder needs a `data.yaml` with `names:`. If it doesn't have one, write it yourself.

## 2. Merge into AURALIS classes
This maps names like *Potholes* to `pothole` and *Manhole* to `open_drain`, and converts segmentation polygons to boxes. Check the **boxes per class** table it prints.

In [ ]:
!python ml/merge_datasets.py {' '.join(sources)}

## 3. Auto-label people, vehicles and animals
A larger pre-trained model adds boxes for person, car, motorbike, dog, cow and so on, so the new model keeps recognising them.

In [ ]:
!python ml/train_yolo.py pseudo

## 4. Train
60 epochs. It stops early if the model stops improving.

In [ ]:
!python ml/train_yolo.py train 60

## 5. Results for your paper

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    print(f); display(Image(f'runs/auralis/{f}', width=900))

In [ ]:
from ultralytics import YOLO
m = YOLO('runs/auralis/weights/best.pt').val(data='runs/data_resolved.yaml', imgsz=320)
for i, n in m.names.items():
    if i < len(m.box.maps):
        print(f'{n:15s} mAP50-95 {m.box.maps[i]:.3f}')
print(f'ALL  mAP50 {m.box.map50:.3f}  precision {m.box.mp:.3f}  recall {m.box.mr:.3f}')

## 6. Export for the Raspberry Pi 4 and download
Unzip `auralis_model.zip` into `pi/models/` on the Pi and restart `server.py`.

In [ ]:
!python ml/train_yolo.py export
!cd pi/models && zip -qr /content/auralis_model.zip auralis_yolo.pt auralis_yolo_ncnn_model
from google.colab import files
files.download('/content/auralis_model.zip')